In [ ]:
import os 
os.chdir("/hpc/home/ephdh/workspace/suzhou_false_validation/analysis")
import sys
sys.path.append("/hpc/home/ephdh/workspace/mammo_foundation/baseline/src/utils")

import ast
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.metrics import roc_auc_score, roc_curve, classification_report, confusion_matrix, accuracy_score

from analysis_utils import get_subgroup_df, generate_roc_curve, thres_eval_metric, pred_eval_metric


In [ ]:
meta_data = pd.read_csv(
        "/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_final_sampled.csv", 
        encoding="utf-16",  
        converters={"DICOMPaths": ast.literal_eval}
        )
meta_data['DICOM_paths'] = meta_data['DICOM_paths'].apply(ast.literal_eval)

In [ ]:
meta_data.head()

In [ ]:
clip_results = pd.read_csv(
    "/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/clip_results.csv", 
    # encoding="utf-16"
    )

clip_results = clip_results.drop_duplicates().reset_index(drop=True)

In [ ]:
clip_results.head()

In [ ]:
# clip_results[clip_results['img_path'].duplicated(keep=False)]

In [ ]:
print(clip_results.img_path.nunique(), clip_results.shape)

In [ ]:
# clip_results['img_path'][0]

In [ ]:
cnn_results = pd.read_csv(
    "/hpc/home/ephdh/workspace/mammo_foundation/analysis/suzhou_test/suzhou_test-image-fold1.csv", 
    # encoding="utf-16"
    )

In [ ]:
cnn_results.head()

In [ ]:
print(clip_results.shape, cnn_results.shape)

In [ ]:
merged_results = pd.merge(clip_results, cnn_results, 
                          left_on='img_path', 
                          right_on='NameList', 
                          how='inner')
merged_results = merged_results.rename(columns={
    'scores': 'clip_image_score',
    'Scores': 'cnn_image_score'})

In [ ]:
print(merged_results.shape, merged_results['img_path'].nunique())

In [ ]:
merged_results.head()

In [ ]:
clip_score_list = []
cnn_score_list = []
patient_clip_score_list = []
patient_cnn_score_list = []
patient_ensemble_score_list = []

for idx, row in tqdm(meta_data.iterrows(), total=meta_data.shape[0]):
    dicom_paths = row['DICOM_paths']['DICOMPaths']
    

    patient_clip_scores = []
    patient_cnn_scores = []
    for dicom_path in dicom_paths:
        image_path = dicom_path.replace("/data2/dh/MammographyData/SzOriginalFinal_2/SzOriginalCleaned", 
                                        "/hpc/home/ephdh/workspace/suzhou_false_validation/data/preprocessed_data").replace('.dcm', '.png')
        result_row = merged_results[merged_results['img_path'] == image_path]
        assert result_row.shape[0]==1, f"No results for image path: {image_path}"

        clip_image_score = result_row['clip_image_score'].values[0]
        cnn_image_score = result_row['cnn_image_score'].values[0]
        
        patient_clip_scores.append(clip_image_score)
        patient_cnn_scores.append(cnn_image_score)
    
    patient_clip_score = max(patient_clip_scores)
    patient_cnn_score = max(patient_cnn_scores)
    
    patient_clip_score_list.append(patient_clip_score)
    patient_cnn_score_list.append(patient_cnn_score)
    clip_score_list.append(patient_clip_scores)
    cnn_score_list.append(patient_cnn_scores)
    patient_ensemble_score = np.mean([patient_clip_score, patient_cnn_score])
    patient_ensemble_score_list.append(patient_ensemble_score)

meta_data['clip_image_scores'] = clip_score_list
meta_data['cnn_image_scores'] = cnn_score_list
meta_data['clip_patient_score'] = patient_clip_score_list
meta_data['cnn_patient_score'] = patient_cnn_score_list
meta_data['ensemble_patient_score'] = patient_ensemble_score_list

In [ ]:
meta_data.head()

In [ ]:
test_df = meta_data[meta_data['Group'].isin(['TN', 'TP'])].reset_index(drop=True)

all_scores = test_df.clip_patient_score.to_numpy()
all_labels = test_df.GroundTruth.to_numpy()

fpr, tpr, thresholds, optimal_threshold, aucs = generate_roc_curve(y_true=all_labels, y_score=all_scores, 
                                                             interpolation=False, drop_intermediate=False, 
                                                             sensitivity_target=None, 
                                                             specificity_target=None, 
                                                             method='youden', 
                                                             return_auc=True)
print(aucs)
thres_eval_metric(y_true=all_labels, y_score=all_scores, threshold=optimal_threshold)

In [ ]:
test_df = meta_data[meta_data['Group'].isin(['TN', 'TP'])].reset_index(drop=True)

all_scores = test_df.cnn_patient_score.to_numpy()
all_labels = test_df.GroundTruth.to_numpy()

fpr, tpr, thresholds, optimal_threshold, aucs = generate_roc_curve(y_true=all_labels, y_score=all_scores, 
                                                             interpolation=False, drop_intermediate=False, 
                                                             sensitivity_target=None, 
                                                             specificity_target=None, 
                                                             method='youden', 
                                                             return_auc=True)
print(aucs)
thres_eval_metric(y_true=all_labels, y_score=all_scores, threshold=optimal_threshold)

In [ ]:
AI_THRESHOLD = 0.536
test_df = meta_data[meta_data['Group'].isin(['TN', 'TP'])].reset_index(drop=True)

all_scores = test_df.ensemble_patient_score.to_numpy()
all_labels = test_df.GroundTruth.to_numpy()

fpr, tpr, thresholds, _, aucs = generate_roc_curve(y_true=all_labels, y_score=all_scores, 
                                                             interpolation=False, drop_intermediate=False, 
                                                             sensitivity_target=None, 
                                                             specificity_target=None, 
                                                             method='youden', 
                                                             return_auc=True)
print(aucs)
print(f'Performance at locked ensemble_patient_score threshold: {AI_THRESHOLD:.3f}')
thres_eval_metric(y_true=all_labels, y_score=all_scores, threshold=AI_THRESHOLD)

In [ ]:
# The rank-average ensemble has been retired. The only ensemble scale used
# downstream is the direct patient-level arithmetic mean defined above.
expected_patient_ensemble = (
    meta_data['clip_patient_score'].astype(float)
    + meta_data['cnn_patient_score'].astype(float)
) / 2
assert np.allclose(
    meta_data['ensemble_patient_score'].astype(float),
    expected_patient_ensemble,
    equal_nan=True,
), 'ensemble_patient_score is not the arithmetic mean of CLIP and CNN patient scores'

In [ ]:
df = meta_data.copy()

In [ ]:
df[['clip_patient_score', 'cnn_patient_score', 'ensemble_patient_score']].head()

In [ ]:
# Rank-based ensemble evaluation removed. The patient-level ensemble is
# evaluated above and is the only ensemble score saved downstream.

In [ ]:
import pandas as pd
import numpy as np

df = meta_data.copy()

# ---- Create y_true from Group ----
df["cancer"] = df["Group"].isin(["TP", "FN"]).astype(int)

# ---- Radiologist binary prediction ----
df["rad_pos"] = df["Group"].isin(["TP", "FP"]).astype(int)

# ---- AI score ----
df["ai_score"] = df["ensemble_patient_score"].astype(float)

# ---- AI binary classification ----
AI_THRESHOLD = 0.536  # modify if needed
df["ai_pos"] = (df["ai_score"] >= AI_THRESHOLD).astype(int)

In [ ]:
def derive_outcomes(df):
    df = df.copy()
    y_true = df["cancer"].values
    rad = df["rad_pos"].values
    ai  = df["ai_pos"].values

    def outcome(pred, truth):
        if pred == 1 and truth == 1: return "TP"
        if pred == 0 and truth == 0: return "TN"
        if pred == 1 and truth == 0: return "FP"
        if pred == 0 and truth == 1: return "FN"

    df["rad_outcome"] = [outcome(r, y) for r, y in zip(rad, y_true)]
    df["ai_outcome"]  = [outcome(a, y) for a, y in zip(ai,  y_true)]
    return df

df = derive_outcomes(df)

In [ ]:
df.head()

In [ ]:
df.to_csv(
    "/hpc/home/ephdh/workspace/suzhou_false_validation/data/meta_data/meta_data_with_model_scores.csv", 
    index=False, 
    encoding="utf-16"
    )